# U2-05 · Regularización, redes convolucionales, recurrentes y autocodificador

*Programación de Redes Neuronales - Machine Learning (IFCD48) · Unidad 2, apartados 2.6 a 2.9*

Ejecuta cada celda con **Mayús + Intro** (en Colab, también con el botón ▶). Con GPU (*Entorno de ejecución → Cambiar tipo de entorno de ejecución*) todo va bastante más rápido; sin ella, varias celdas tardan varios minutos cada una.

> Datos: Fashion-MNIST, de Zalando Research (licencia MIT), y MNIST, de Y. LeCun, C. Cortes y C. J. C. Burges (CC BY-SA 3.0). Keras los descarga en la primera ejecución.

## 1. Regularización y matriz de confusión (2.6)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers

(x, y), (x_pru, y_pru) = keras.datasets.fashion_mnist.load_data()
x, x_pru = x / 255.0, x_pru / 255.0
x_ent, y_ent = x[:50_000], y[:50_000]          # entrenamiento
x_val, y_val = x[50_000:], y[50_000:]          # validación

def red(extra=(), drop=0.0, l2=0.0):
    keras.utils.set_random_seed(0)
    reg = keras.regularizers.L2(l2) if l2 else None
    capas = [keras.Input(shape=(28, 28, 1)), *extra, layers.Flatten()]
    for n in (256, 128):
        capas += [layers.Dense(n, activation="relu",
                               kernel_regularizer=reg),
                  layers.Dropout(drop)]
    m = keras.Sequential(capas + [layers.Dense(10, "softmax")])
    m.compile("adam", "sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m

aumento = (layers.RandomFlip("horizontal"),       # solo al entrenar
           layers.RandomTranslation(0.1, 0.1))
variantes = {"sin regularizar": red(), "dropout 0.3": red(drop=0.3),
             "L2 1e-4": red(l2=1e-4),
             "aumento de datos": red(aumento)}
parada = keras.callbacks.EarlyStopping(
    patience=5, restore_best_weights=True)
for nombre, m in variantes.items():
    h = m.fit(x_ent, y_ent, epochs=60, batch_size=128, verbose=0,
              validation_data=(x_val, y_val), callbacks=[parada])
    b = int(np.argmin(h.history["val_loss"]))
    print(f"{nombre:<17} épocas {len(h.history['loss']):2d}  "
          f"entrenamiento {h.history['accuracy'][b]:.3f}  "
          f"validación {h.history['val_accuracy'][b]:.3f}")

m = variantes["dropout 0.3"]                    # la elegida
pred = m.predict(x_pru, verbose=0).argmax(1)
print(f"dropout 0.3 en prueba: {np.mean(pred == y_pru):.4f}")
conf = np.zeros((10, 10), int)
np.add.at(conf, (y_pru, pred), 1)               # matriz de confusión
print("camisa confundida con:", conf[6])
plt.imshow(conf, cmap="Blues")
for i in range(10):
    for j in range(10):
        plt.text(j, i, conf[i, j], ha="center", va="center",
                 fontsize=5, color="w" if conf[i, j] > 500 else "k")
plt.xlabel("clase predicha"); plt.ylabel("clase real")
plt.xticks(range(10)); plt.yticks(range(10))
plt.show()  # plt.savefig("confusion.pdf")

**Prueba.** Sube el dropout a 0.5. ¿Mejora la validación o la red empieza a quedarse corta?

## 2. Qué hacen L1 y L2 con los pesos (2.6)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import regularizers

(x, y), (xp, yp) = keras.datasets.fashion_mnist.load_data()
x, xp = x / 255.0, xp / 255.0
fig, ejes = plt.subplots(1, 3, figsize=(4.4, 1.5), sharey=True,
                         layout="constrained")
for eje, (nombre, reg) in zip(ejes, (
        ("sin regularizar", None),
        ("L2", regularizers.L2(1e-3)),
        ("L1", regularizers.L1(1e-4)))):
    keras.utils.set_random_seed(0)
    red = keras.Sequential([
        keras.Input(shape=(28, 28)), keras.layers.Flatten(),
        keras.layers.Dense(128, activation="relu",
                           kernel_regularizer=reg),
        keras.layers.Dense(10, activation="softmax")])
    red.compile("adam", "sparse_categorical_crossentropy",
                metrics=["accuracy"])
    red.fit(x, y, epochs=10, batch_size=128, verbose=0)
    W = red.layers[1].get_weights()[0].ravel()
    a = red.evaluate(xp, yp, verbose=0)[1]
    print(f"{nombre:<15} prueba {a:.4f}, |w| medio "
          f"{np.abs(W).mean():.4f}, casi nulos (< 0,001) "
          f"{np.mean(np.abs(W) < 1e-3):.0%}")
    eje.hist(W, bins=80, range=(-0.3, 0.3)); eje.set_title(nombre)
plt.show()  # fig.savefig("pesos_reg.pdf")

## 3. Umbrales por clase (2.6)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from sklearn.metrics import (precision_recall_curve,
                             average_precision_score)

(x, y), (xp, yp) = keras.datasets.fashion_mnist.load_data()
x, xp = x / 255.0, xp / 255.0
keras.utils.set_random_seed(0)
red = keras.Sequential([
    keras.Input(shape=(28, 28)), keras.layers.Flatten(),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dense(10, activation="softmax")])
red.compile("adam", "sparse_categorical_crossentropy",
            metrics=["accuracy"])
red.fit(x, y, epochs=10, batch_size=128, verbose=0)
p = red.predict(xp, verbose=0)

plt.figure(figsize=(4.4, 2.2))
for c, nombre in ((6, "camisa"), (1, "pantalón"), (0, "camiseta")):
    prec, sens, umbral = precision_recall_curve(yp == c, p[:, c])
    pm = average_precision_score(yp == c, p[:, c])
    print(f"{nombre:<9} precisión media {pm:.3f}")
    plt.plot(sens, prec, label=nombre)
    i = np.argmin(np.abs(sens - 0.8))        # 80 % de sensibilidad
    u = umbral[min(i, len(umbral) - 1)]
    print(f"   para detectar el 80 %: umbral {u:.3f}, "
          f"precisión {prec[i]:.3f}")
plt.xlabel("sensibilidad"); plt.ylabel("precisión"); plt.legend()
plt.show()  # plt.savefig("pr_clases.pdf")

## 4. Clases desequilibradas (2.6)

In [ ]:
import numpy as np
import keras

(x, y), (xp, yp) = keras.datasets.fashion_mnist.load_data()
x, xp = x / 255.0, xp / 255.0
rng = np.random.default_rng(0)
camisas = np.flatnonzero(y[:50_000] == 6)
fuera = rng.choice(camisas, int(0.95 * len(camisas)), replace=False)
sel = np.setdiff1d(np.arange(50_000), fuera)      # 5 % de camisas
print("camisas en entrenamiento:", np.sum(y[sel] == 6), "de",
      len(sel))

for pesos in (None, "equilibrados"):
    keras.utils.set_random_seed(0)
    red = keras.Sequential([
        keras.Input(shape=(28, 28)), keras.layers.Flatten(),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dense(10, activation="softmax")])
    red.compile("adam", "sparse_categorical_crossentropy",
                metrics=["accuracy"])
    n = np.bincount(y[sel])
    cw = (None if pesos is None              # peso inverso al tamaño
          else dict(enumerate(n.sum() / (10 * n))))
    red.fit(x[sel], y[sel], epochs=10, batch_size=128, verbose=0,
            class_weight=cw)
    p = red.predict(xp, verbose=0).argmax(1)
    sens = np.mean(p[yp == 6] == 6)
    prec = np.mean(yp[p == 6] == 6)
    print(f"pesos {pesos or 'iguales':<12}: acierto "
          f"{np.mean(p == yp):.4f}, camisa: sensibilidad {sens:.3f},"
          f" precisión {prec:.3f}")

## 5. ¿Son fiables las probabilidades? (2.6)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from scipy.optimize import minimize_scalar
from scipy.special import softmax, log_softmax
from sklearn.metrics import precision_recall_fscore_support

(x, y), (xp, yp) = keras.datasets.fashion_mnist.load_data()
x, xp = x / 255.0, xp / 255.0
keras.utils.set_random_seed(0)
red = keras.Sequential([
    keras.Input(shape=(28, 28)), keras.layers.Flatten(),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dense(10)])                     # sin softmax: logits
red.compile("adam", keras.losses.SparseCategoricalCrossentropy(
    from_logits=True), metrics=["accuracy"])
red.fit(x[:50_000], y[:50_000], epochs=30, batch_size=128, verbose=0)
zv = red.predict(x[50_000:], verbose=0)     # validación
zp = red.predict(xp, verbose=0)             # prueba

def ece(prob, y, bins=10):          # error de calibración esperado
    conf, ok = prob.max(1), prob.argmax(1) == y
    tramos = np.minimum((conf * bins).astype(int), bins - 1)
    return sum(abs(conf[tramos == b].mean() - ok[tramos == b].mean())
               * (tramos == b).mean() for b in range(bins)
               if (tramos == b).any())

nll = lambda T: -log_softmax(zv / T, 1)[np.arange(10_000),
                                        y[50_000:]].mean()
T = minimize_scalar(nll, bounds=(0.5, 10), method="bounded").x
p1, pT = softmax(zp, 1), softmax(zp / T, 1)
print(f"acierto en prueba {np.mean(p1.argmax(1) == yp):.4f}; "
      f"confianza media {p1.max(1).mean():.4f}")
print(f"ECE sin calibrar {ece(p1, yp):.4f}; temperatura T = {T:.2f};"
      f" ECE calibrada {ece(pT, yp):.4f}")
pr, se, f1, _ = precision_recall_fscore_support(yp, p1.argmax(1))
print("F1 por clase:", f1.round(2), "| media", f1.mean().round(3))

fig, eje = plt.subplots(figsize=(2.8, 2.4))
for p, nombre in ((p1, "sin calibrar"), (pT, f"T = {T:.2f}")):
    conf, ok = p.max(1), p.argmax(1) == yp
    tr = np.minimum((conf * 10).astype(int), 9)
    b = [b for b in range(10) if (tr == b).sum() > 20]
    eje.plot([conf[tr == i].mean() for i in b],
             [ok[tr == i].mean() for i in b], "o-", ms=3,
             label=nombre)
eje.plot([0, 1], [0, 1], color="0.6", lw=0.8)
eje.set_xlabel("confianza"); eje.set_ylabel("acierto real")
eje.legend(); plt.show()  # fig.savefig("calibrar.pdf")

## 6. La convolución a mano (2.7)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras

(x, y), _ = keras.datasets.fashion_mnist.load_data()
print("clase de la imagen:", y[0])        # 9 = botín
img = x[0] / 255.0             # la primera imagen: un botín

def convolucion(img, filtro):
    """Desliza el filtro por la imagen (sin relleno, paso 1)."""
    k = filtro.shape[0]
    alto, ancho = img.shape[0] - k + 1, img.shape[1] - k + 1
    sal = np.zeros((alto, ancho))
    for i in range(alto):
        for j in range(ancho):
            sal[i, j] = np.sum(img[i:i + k, j:j + k] * filtro)
    return sal

def max_pooling(m, t=2):
    a, b = m.shape[0] // t, m.shape[1] // t
    return m[:a * t, :b * t].reshape(a, t, b, t).max(axis=(1, 3))

vertical = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]])  # Sobel
horizontal = vertical.T
bv, bh = convolucion(img, vertical), convolucion(img, horizontal)
rp = max_pooling(np.maximum(bh, 0))         # ReLU y reducción
print("imagen", img.shape, "-> convolución", bv.shape,
      "-> pooling", rp.shape)
print(f"respuesta máxima: vertical {np.abs(bv).max():.2f}, "
      f"horizontal {np.abs(bh).max():.2f}")
print("pesos de 32 filtros 3x3:", 32 * (3 * 3 + 1),
      "| pesos de 256 neuronas densas:", 256 * (784 + 1))

fig, ejes = plt.subplots(1, 4, figsize=(4.4, 1.3))
for e, m, t in zip(ejes, [img, bv, bh, rp],
                   ["imagen", "bordes vert.",
                    "bordes horiz.", "ReLU y pooling"]):
    if m is img:
        e.imshow(m, cmap="gray_r")
    else:                                   # 0 en blanco
        e.imshow(m, cmap="RdBu", vmin=-4, vmax=4)
    e.set_title(t, fontsize=6); e.axis("off")
plt.show()  # fig.savefig("convolucion.pdf")

## 7. Una red convolucional (2.7)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers

(x, y), (x_pru, y_pru) = keras.datasets.fashion_mnist.load_data()
x, x_pru = x[..., None] / 255.0, x_pru[..., None] / 255.0  # 1 canal

keras.utils.set_random_seed(0)
red = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, activation="relu"),     # 32 filtros 3x3
    layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(2),
    layers.Flatten(), layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax")])
red.compile("adam", "sparse_categorical_crossentropy",
            metrics=["accuracy"])
red.summary()
parada = keras.callbacks.EarlyStopping(
    patience=3, restore_best_weights=True)
h = red.fit(x[:50_000], y[:50_000], epochs=30, batch_size=128,
            validation_data=(x[50_000:], y[50_000:]),
            callbacks=[parada], verbose=0)
print(f"épocas {len(h.history['loss'])}; prueba "
      f"{red.evaluate(x_pru, y_pru, verbose=0)[1]:.4f}")
pred = red.predict(x_pru, verbose=0).argmax(1)
print(f"camisa: {np.mean(pred[y_pru == 6] == 6):.3f}")

# qué ve la primera capa: 8 de sus 32 mapas para el primer botín
capa1 = keras.Model(red.inputs, red.layers[0].output)
mapas = capa1.predict(x[:1], verbose=0)[0]       # (26, 26, 32)
fig, ejes = plt.subplots(1, 8, figsize=(4.4, 0.75))
for k, e in enumerate(ejes):
    e.imshow(mapas[..., k]); e.axis("off")
plt.show()  # fig.savefig("mapas.pdf")

## 8. Redes más profundas: las conexiones residuales (2.7)

Sin GPU tarda más de diez minutos.

In [ ]:
import keras
from keras import layers

(x, y), (xp, yp) = keras.datasets.fashion_mnist.load_data()
x, xp = x[..., None] / 255.0, xp[..., None] / 255.0

def red(residual, bloques=8):
    e = keras.Input(shape=(28, 28, 1))
    h = layers.Conv2D(32, 3, padding="same", activation="relu")(e)
    for _ in range(bloques):            # 16 convoluciones más
        f = layers.Conv2D(32, 3, padding="same",
                          activation="relu")(h)
        f = layers.Conv2D(32, 3, padding="same")(f)
        h = layers.Activation("relu")(h + f if residual else f)
    h = layers.GlobalAveragePooling2D()(h)
    return keras.Model(e, layers.Dense(10, "softmax")(h))

for residual in (False, True):
    keras.utils.set_random_seed(0)
    m = red(residual)
    m.compile(keras.optimizers.Adam(1e-3),
              "sparse_categorical_crossentropy",
              metrics=["accuracy"])
    h = m.fit(x[:20_000], y[:20_000], epochs=5, batch_size=128,
              verbose=0)
    ent = [round(a, 3) for a in h.history["accuracy"]]
    print(f"residual {residual!s:<5}: {m.count_params()} parámetros,"
          f" entrenamiento por época {ent}"
          f", prueba {m.evaluate(xp, yp, verbose=0)[1]:.4f}")

## 9. Parámetros, filtros y saliencia con la red del proyecto (2.7)

La primera celda descarga la red convolucional del proyecto de la web del libro.

In [ ]:
import urllib.request              # la red del proyecto (apartado 2.12)
urllib.request.urlretrieve("https://especialidades-formativas.github.io/ifcd48/recursos/u2/app/mnist_cnn.keras", "mnist_cnn.keras")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras

red = keras.models.load_model("mnist_cnn.keras")   # la del proyecto
for c in red.layers:
    print(f"{c.name:<16} {str(c.output.shape):<18} "
          f"parámetros {c.count_params():6d}")
densa = 784 * 16 * 26 * 26           # misma salida, sin compartir
print(f"una capa densa con la misma salida que la primera "
      f"convolución: {densa:,} pesos")
W = red.layers[0].get_weights()[0][:, :, 0, :]    # 3x3x16
fig, ejes = plt.subplots(2, 8, figsize=(4.4, 1.2))
for k, e in enumerate(ejes.flat):
    e.imshow(W[:, :, k], cmap="RdBu", vmin=-np.abs(W).max(),
             vmax=np.abs(W).max())
    e.axis("off")
plt.show()  # fig.savefig("filtros.pdf")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

red = keras.models.load_model("mnist_cnn.keras")   # la del proyecto
(_, _), (xp, yp) = keras.datasets.mnist.load_data()
x = tf.constant(xp[:8, ..., None] / 255.0, tf.float32)
with tf.GradientTape() as cinta:
    cinta.watch(x)
    p = red(x)
    elegida = tf.reduce_max(p, axis=1)       # prob. de su respuesta
g = np.abs(cinta.gradient(elegida, x).numpy()[..., 0])
tinta = xp[:8] > 0
print("predicciones:", p.numpy().argmax(1), "reales:", yp[:8])
print("parte de la sensibilidad en píxeles con tinta:",
      np.round([g[i][tinta[i]].sum() / g[i].sum() for i in range(8)],
               2))
print("parte de la imagen con tinta:            ",
      tinta.mean(axis=(1, 2)).round(2))

fig, ejes = plt.subplots(2, 8, figsize=(4.4, 1.2))
for i in range(8):
    ejes[0, i].imshow(xp[i]); ejes[1, i].imshow(g[i], cmap="hot_r")
    for e in ejes[:, i]:
        e.axis("off")
plt.show()  # fig.savefig("saliencia.pdf")

**Prueba.** Añade a esta red el aumento de datos de la celda 1 (`RandomFlip` y `RandomTranslation`) justo después de `keras.Input`. ¿Ayuda ahora, con una red que sí aprovecha la forma de la imagen?

## 10. Aumento de datos con una red convolucional (2.7)

Tarda bastante sin GPU: entrena tres redes.

In [ ]:
import keras
from keras import layers

(x, y), (xp, yp) = keras.datasets.fashion_mnist.load_data()
x, xp = x[..., None] / 255.0, xp[..., None] / 255.0
n = 10_000                         # pocos datos: donde más ayuda

def cnn(aumento, epocas):
    keras.utils.set_random_seed(0)
    extra = [layers.RandomFlip("horizontal"),
             layers.RandomTranslation(0.1, 0.1)] if aumento else []
    red = keras.Sequential([
        keras.Input(shape=(28, 28, 1)), *extra,  # solo al entrenar
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(), layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax")])
    red.compile("adam", "sparse_categorical_crossentropy",
                metrics=["accuracy"])
    h = red.fit(x[:n], y[:n], epochs=epocas, batch_size=128,
                validation_data=(x[50_000:], y[50_000:]), verbose=0)
    return red, h.history

for aumento, epocas in ((False, 30), (True, 30), (True, 90)):
    red, h = cnn(aumento, epocas)
    print(f"aumento {aumento!s:<5} {epocas} épocas: entrenamiento "
          f"{h['accuracy'][-1]:.3f}, mejor validación "
          f"{max(h['val_accuracy']):.4f}, prueba "
          f"{red.evaluate(xp, yp, verbose=0)[1]:.4f}")

## 11. Transferir lo aprendido (2.7)

In [ ]:
import keras
from keras import layers

(fx, fy), _ = keras.datasets.fashion_mnist.load_data()
(mx, my), (mxp, myp) = keras.datasets.mnist.load_data()
fx, mx, mxp = (a[..., None] / 255.0 for a in (fx, mx, mxp))

def base():
    return keras.Sequential([
        keras.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten()])

keras.utils.set_random_seed(0)
origen = base()                      # 1) aprende con 60 000 prendas
modelo = keras.Sequential([origen, layers.Dense(10, "softmax")])
modelo.compile("adam", "sparse_categorical_crossentropy")
modelo.fit(fx, fy, epochs=5, batch_size=128, verbose=0)

for n in (100, 500, 2000):           # 2) pocos dígitos etiquetados
    for nombre, b in (("desde cero", base()),
                      ("transferida", origen)):
        keras.utils.set_random_seed(0)
        b.trainable = nombre == "desde cero"   # congela la base
        red = keras.Sequential([b, layers.Dense(10, "softmax")])
        red.compile("adam", "sparse_categorical_crossentropy",
                    metrics=["accuracy"])
        red.fit(mx[:n], my[:n], epochs=30, batch_size=32, verbose=0)
        print(f"{n:5d} dígitos, {nombre:<11}: prueba "
              f"{red.evaluate(mxp, myp, verbose=0)[1]:.4f}")

## 12. Memoria larga: el problema de la suma (2.8)

In [ ]:
import numpy as np
import keras
from keras import layers

def sumas(n, largo, semilla):
    """Problema de la suma: hay que sumar los dos valores marcados,
    uno en la primera mitad de la secuencia y otro en la segunda."""
    rng = np.random.default_rng(semilla)
    v = rng.uniform(0, 1, (n, largo))
    m = np.zeros((n, largo))
    m[np.arange(n), rng.integers(0, largo // 2, n)] = 1
    m[np.arange(n), rng.integers(largo // 2, largo, n)] = 1
    x = np.stack([v, m], axis=-1).astype("float32")
    return x, (v * m).sum(1).astype("float32")

if __name__ == "__main__":          # solo al ejecutar este archivo
    for largo in [10, 50, 100]:
        x, y = sumas(10_000, largo, 0)
        xp, yp = sumas(1000, largo, 1)
        for tipo in [layers.SimpleRNN, layers.LSTM]:
            keras.utils.set_random_seed(0)
            red = keras.Sequential([keras.Input(shape=(largo, 2)),
                                    tipo(32), layers.Dense(1)])
            red.compile("adam", "mse")
            red.fit(x, y, epochs=20, batch_size=64, verbose=0)
            ecm = red.evaluate(xp, yp, verbose=0)
            print(f"{largo:3d} pasos, {tipo.__name__:<9}: error "
                  f"{ecm:.4f}")
    print(f"sin memoria (responder siempre 1): "
          f"{np.mean((yp - 1) ** 2):.4f}")

**Prueba.** Cambia `layers.LSTM` por `layers.GRU`. ¿Resuelve también las secuencias de 100 pasos?

## 13. Imágenes como secuencias y series temporales (2.8)

In [ ]:
import numpy as np
import keras
from keras import layers

(x, y), (x_pru, y_pru) = keras.datasets.fashion_mnist.load_data()
x, x_pru = x / 255.0, x_pru / 255.0     # (n, 28, 28): 28 pasos de 28

keras.utils.set_random_seed(0)
red = keras.Sequential([keras.Input(shape=(28, 28)),
                        layers.LSTM(128),        # lee fila a fila
                        layers.Dense(10, "softmax")])
red.compile("adam", "sparse_categorical_crossentropy",
            metrics=["accuracy"])
parada = keras.callbacks.EarlyStopping(
    patience=3, restore_best_weights=True)
h = red.fit(x[:50_000], y[:50_000], epochs=30, batch_size=128,
            validation_data=(x[50_000:], y[50_000:]),
            callbacks=[parada], verbose=0)
print(f"parámetros {red.count_params():,}; "
      f"épocas {len(h.history['loss'])}")
print(f"prueba: {red.evaluate(x_pru, y_pru, verbose=0)[1]:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers

rng = np.random.default_rng(0)
t = np.arange(3000)
serie = (np.sin(2 * np.pi * t / 50) + 0.5 * np.sin(2 * np.pi * t / 7)
         + rng.normal(0, 0.2, t.size)).astype("float32")

def ventanas(s, n=30):
    """Cada ejemplo: n valores seguidos; objetivo: el siguiente."""
    x = np.stack([s[i:i + n] for i in range(len(s) - n)])
    return x[..., None], s[n:]

x, y = ventanas(serie[:2500])           # pasado: entrenamiento
xp, yp = ventanas(serie[2500:])          # futuro: prueba
keras.utils.set_random_seed(0)
red = keras.Sequential([keras.Input(shape=(30, 1)),
                        layers.LSTM(32), layers.Dense(1)])
red.compile("adam", "mse")
red.fit(x, y, epochs=20, batch_size=32, verbose=0)
pred = red.predict(xp, verbose=0)[:, 0]
print(f"error medio, repetir el último valor: "
      f"{np.mean(np.abs(xp[:, -1, 0] - yp)):.3f}")
print(f"error medio, LSTM: {np.mean(np.abs(pred - yp)):.3f}")
print(f"mínimo posible, por el ruido: {0.2 * np.sqrt(2 / np.pi):.3f}")

plt.plot(yp[:150], ".", ms=2, label="real")
plt.plot(pred[:150], label="LSTM")
plt.xlabel("paso"); plt.legend(); plt.show()  # plt.savefig("serie.pdf")

## 14. Recurrentes bidireccionales y apiladas (2.8)

In [ ]:
import time
import keras
from keras import layers

(x, y), (xp, yp) = keras.datasets.fashion_mnist.load_data()
x, xp = x / 255.0, xp / 255.0           # 28 filas de 28 píxeles
variantes = {
    "LSTM": [layers.LSTM(64)],
    "LSTM bidireccional": [layers.Bidirectional(layers.LSTM(64))],
    "dos LSTM apiladas": [layers.LSTM(64, return_sequences=True),
                          layers.LSTM(64)]}
for nombre, capas in variantes.items():
    keras.utils.set_random_seed(0)
    red = keras.Sequential([keras.Input(shape=(28, 28)), *capas,
                            layers.Dense(10, activation="softmax")])
    red.compile("adam", "sparse_categorical_crossentropy",
                metrics=["accuracy"])
    t = time.perf_counter()
    red.fit(x, y, epochs=8, batch_size=128, verbose=0)
    print(f"{nombre:<19} {red.count_params():6d} parámetros, "
          f"{time.perf_counter() - t:4.0f} s, prueba "
          f"{red.evaluate(xp, yp, verbose=0)[1]:.4f}")

## 15. Comparar las capas recurrentes (2.8)

In [ ]:
import time
import numpy as np
import keras
from keras import layers

rng = np.random.default_rng(0)                 # la serie de x_serie
t = np.arange(3000)
serie = (np.sin(2 * np.pi * t / 50) + 0.5 * np.sin(2 * np.pi * t / 7)
         + rng.normal(0, 0.2, t.size)).astype("float32")
v = lambda s, n=30: (np.stack([s[i:i + n] for i in range(len(s) - n)]
                              )[..., None], s[n:])
x, y = v(serie[:2500]); xp, yp = v(serie[2500:])

for capa in (layers.SimpleRNN, layers.GRU, layers.LSTM,
             layers.Conv1D):
    keras.utils.set_random_seed(0)
    nucleo = ([capa(32, 5, activation="relu"), layers.Flatten()]
              if capa is layers.Conv1D else [capa(32)])
    red = keras.Sequential([keras.Input(shape=(30, 1)), *nucleo,
                            layers.Dense(1)])
    red.compile("adam", "mse")
    t0 = time.perf_counter()
    red.fit(x, y, epochs=20, batch_size=32, verbose=0)
    error = np.mean(np.abs(red.predict(xp, verbose=0)[:, 0] - yp))
    print(f"{capa.__name__:<9} {red.count_params():5d} parámetros, "
          f"{time.perf_counter() - t0:3.0f} s, error {error:.3f}")

## 16. Recomendar con una red (2.9)

La primera celda descarga MovieLens 100K de la web de GroupLens (no se puede redistribuir).

In [ ]:
import urllib.request, zipfile        # MovieLens 100K, de GroupLens
urllib.request.urlretrieve("https://files.grouplens.org/datasets/movielens/ml-100k.zip", "ml-100k.zip")
zipfile.ZipFile("ml-100k.zip").extractall()

In [ ]:
import numpy as np
import pandas as pd
import keras
from keras import layers

notas = pd.read_csv("ml-100k/u.data", sep="\t",
                    names=["usuario", "peli", "nota", "fecha"])
rng = np.random.default_rng(0)
prueba = rng.random(len(notas)) < 0.2       # el reparto de la u. 1
ent, pru = notas[~prueba], notas[prueba]

def entradas(d):
    return [d.usuario.values - 1, d.peli.values - 1]

u, p = keras.Input(shape=(1,)), keras.Input(shape=(1,))
eu = layers.Flatten()(layers.Embedding(943, 16)(u))   # 16 números
ep = layers.Flatten()(layers.Embedding(1682, 16)(p))  # por cada uno
h = layers.Concatenate()([eu, ep])
h = layers.Dense(64, activation="relu")(h)
h = layers.Dropout(0.3)(h)
nota = layers.Dense(1)(h)
keras.utils.set_random_seed(0)
red = keras.Model([u, p], nota)
red.compile("adam", "mse")
red.fit(entradas(ent), ent.nota.values, epochs=10, batch_size=256,
        validation_split=0.1, verbose=0,
        callbacks=[keras.callbacks.EarlyStopping(
            patience=2, restore_best_weights=True)])
pred = red.predict(entradas(pru), verbose=0)[:, 0].clip(1, 5)
rmse = np.sqrt(np.mean((pred - pru.nota.values) ** 2))
media = np.sqrt(np.mean((ent.nota.mean() - pru.nota.values) ** 2))
print(f"RMSE en prueba: red {rmse:.3f}; nota media {media:.3f}")

## 17. Autocodificador y anomalías (2.9)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers

(x, y), (x_pru, y_pru) = keras.datasets.fashion_mnist.load_data()
x, x_pru = x.reshape(-1, 784) / 255, x_pru.reshape(-1, 784) / 255
(_, _), (dig, _) = keras.datasets.mnist.load_data()
dig = dig.reshape(-1, 784) / 255              # dígitos: algo distinto

def autocodificador(k):
    keras.utils.set_random_seed(0)
    cod = keras.Sequential([keras.Input(shape=(784,)),
                            layers.Dense(256, "relu"),
                            layers.Dense(k)])     # código: k números
    dec = keras.Sequential([keras.Input(shape=(k,)),
                            layers.Dense(256, "relu"),
                            layers.Dense(784, "sigmoid")])
    auto = keras.Sequential([cod, dec])
    auto.compile("adam", "binary_crossentropy")
    auto.fit(x, x, epochs=20, batch_size=256, verbose=0)  # x -> x
    return cod, auto

def error(auto, datos):                 # error de reconstrucción
    return np.mean((datos - auto.predict(datos, verbose=0)) ** 2, 1)

for k in [2, 32]:
    cod, auto = autocodificador(k)
    e_ropa, e_dig = error(auto, x_pru), error(auto, dig)
    umbral = np.quantile(e_ropa, 0.99)  # 1 % de falsas alarmas
    print(f"código de {k:2d}: error ropa {e_ropa.mean():.4f}, "
          f"dígitos {e_dig.mean():.4f}; dígitos anómalos "
          f"{np.mean(e_dig > umbral):.3f}")
    if k == 2:
        z = cod.predict(x_pru, verbose=0)
        plt.scatter(z[:, 0], z[:, 1], c=y_pru, cmap="tab10", s=0.3)
        plt.colorbar(ticks=range(10)); plt.show()  # plt.savefig("codigos.pdf")

## 18. Agrupar en el espacio del código (2.9)

In [ ]:
import numpy as np
import keras
from keras import layers
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

(x, _), (xp, yp) = keras.datasets.fashion_mnist.load_data()
x, xp = x.reshape(-1, 784) / 255, xp.reshape(-1, 784) / 255
keras.utils.set_random_seed(0)
cod = keras.Sequential([keras.Input(shape=(784,)),
                        layers.Dense(256, "relu"), layers.Dense(10)])
dec = keras.Sequential([keras.Input(shape=(10,)),
                        layers.Dense(256, "relu"),
                        layers.Dense(784, "sigmoid")])
auto = keras.Sequential([cod, dec])
auto.compile("adam", "binary_crossentropy")
auto.fit(x, x, epochs=20, batch_size=256, verbose=0)

versiones = {"784 píxeles": xp,
             "10 componentes PCA": PCA(10, random_state=0)
                 .fit(x).transform(xp),
             "10 números del código": cod.predict(xp, verbose=0)}
for nombre, datos in versiones.items():
    g = KMeans(10, n_init=10, random_state=0).fit_predict(datos)
    print(f"{nombre:<22} ARI con las prendas reales: "
          f"{adjusted_rand_score(yp, g):.3f}")

## 19. Un autocodificador que limpia imágenes (2.9)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers

(x, _), (xp, _) = keras.datasets.fashion_mnist.load_data()
x, xp = x.reshape(-1, 784) / 255, xp.reshape(-1, 784) / 255
rng = np.random.default_rng(0)
ensucia = lambda a: np.clip(a + rng.normal(0, 0.3, a.shape), 0, 1)
x_r, xp_r = ensucia(x), ensucia(xp)        # imágenes con ruido

keras.utils.set_random_seed(0)
auto = keras.Sequential([keras.Input(shape=(784,)),
                         layers.Dense(256, "relu"),
                         layers.Dense(64, "relu"),
                         layers.Dense(256, "relu"),
                         layers.Dense(784, "sigmoid")])
auto.compile("adam", "binary_crossentropy")
auto.fit(x_r, x, epochs=20, batch_size=256,
         verbose=0)                     # entrada sucia, salida limpia
limpia = auto.predict(xp_r, verbose=0)
ecm = lambda a: np.mean((a - xp) ** 2)
print(f"error con ruido {ecm(xp_r):.4f}; "
      f"tras limpiar {ecm(limpia):.4f}")

fig, ejes = plt.subplots(3, 8, figsize=(4.4, 1.8))
for j in range(8):
    for i, img in enumerate((xp_r, limpia, xp)):
        ejes[i, j].imshow(img[j].reshape(28, 28), vmin=0, vmax=1)
        ejes[i, j].axis("off")
plt.show()  # fig.savefig("ruido.pdf")

## 20. Q-learning con una tabla (2.9)

In [ ]:
import numpy as np

N, META, TRAMPA = 5, (4, 4), (2, 2)        # la cuadrícula de antes
MOV = [(-1, 0), (1, 0), (0, -1), (0, 1)]
def paso(s, a):
    f = min(max(s[0] + MOV[a][0], 0), N - 1)
    c = min(max(s[1] + MOV[a][1], 0), N - 1)
    if (f, c) == META: return (f, c), 10.0, True
    if (f, c) == TRAMPA: return (f, c), -10.0, True
    return (f, c), -1.0, False

for eps in (0.0, 0.1, 0.5):
    rng = np.random.default_rng(0)
    Q = np.zeros((N, N, 4))                 # una casilla por estado
    pasos = []
    for episodio in range(300):
        s, fin, n = (0, 0), False, 0
        while not fin and n < 100:
            a = (rng.integers(4) if rng.random() < eps
                 else int(Q[s].argmax()))
            s2, r, fin = paso(s, a)
            objetivo = r + (0 if fin else 0.9 * Q[s2].max())
            Q[s][a] += 0.5 * (objetivo - Q[s][a])   # Q-learning
            s, n = s2, n + 1
        pasos.append(n)
    s, n = (0, 0), 0                         # recorrido sin explorar
    while s != META and n < 20:
        s, _, _ = paso(s, int(Q[s].argmax())); n += 1
    print(f"exploración {eps}: últimos 100 episodios, "
          f"{np.mean(pasos[-100:]):.1f} pasos de media; camino "
          f"final {n if s == META else 'no llega'}")

## 21. El factor de descuento (2.9)

In [ ]:
import numpy as np

# pasillo: casilla 0 a 9; en la 2 hay una recompensa pequeña (+1)
# que termina el episodio; en la 9, una grande (+10)
def paso(s, a):                       # a = 0 izquierda, 1 derecha
    s2 = max(0, min(9, s + (1 if a else -1)))
    if s2 == 2: return s2, 1.0, True
    if s2 == 9: return s2, 10.0, True
    return s2, 0.0, False

for gamma in (0.5, 0.7, 0.9, 0.99):
    rng = np.random.default_rng(0)
    Q = np.zeros((10, 2))
    for episodio in range(5000):
        s, fin = 3, False             # empieza junto a la pequeña
        while not fin:
            a = (rng.integers(2) if rng.random() < 0.5
                 else int(Q[s].argmax()))
            s2, r, fin = paso(s, a)
            Q[s, a] += 0.1 * (r + (0 if fin else gamma * Q[s2].max())
                              - Q[s, a])
            s = s2
    elige = "derecha (+10)" if Q[3].argmax() else "izquierda (+1)"
    print(f"gamma {gamma}: desde la casilla 3 va a la {elige}")
    print(f"   Q izquierda {Q[3, 0]:.2f} (teoría 1.00), derecha "
          f"{Q[3, 1]:.2f} (teoría {10 * gamma ** 5:.2f})")

**Ejercicio.** Entrena el autocodificador solo con camisetas y busca, entre todas las prendas de prueba, las 10 con más error de reconstrucción. ¿Qué prendas son?

> LeCun, Y., Bottou, L., Bengio, Y. y Haffner, P. (1998). Gradient-based learning applied to document recognition. *Proceedings of the IEEE*, 86(11), 2278–2324. https://doi.org/10.1109/5.726791
>
> Hochreiter, S. y Schmidhuber, J. (1997). Long short-term memory. *Neural Computation*, 9(8), 1735–1780. https://doi.org/10.1162/neco.1997.9.8.1735
>
> Hinton, G. E. y Salakhutdinov, R. R. (2006). Reducing the dimensionality of data with neural networks. *Science*, 313(5786), 504–507. https://doi.org/10.1126/science.1127647